In [1]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


/home/mert/.local/lib/python3.10/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


In [2]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
O   0.000   0.000   1.157
C   0.000   1.094   0.346
C   0.000  -1.094   0.346
C   0.000   0.716  -0.955
C   0.000  -0.716  -0.955
H   0.000   2.047   0.840
H   0.000  -2.047   0.840
H   0.000   1.370  -1.810
H   0.000  -1.370  -1.810'''
molecule = 'C4H4O'
spin = 0
charge =  0


In [6]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 5 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [4]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [7]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

overwrite output file: ./C4H4O.txt
orbital  energy   occ
0     -20.628271116858404    2.0
1     -11.225082727691444    2.0
2     -11.224896606168548    2.0
3     -11.17134215837774    2.0
4     -11.170926791871459    2.0
5     -1.3975941520725916    2.0
6     -1.024644573264391    2.0
7     -0.9451845789255502    2.0
8     -0.7416395171723662    2.0
9     -0.7288623181182812    2.0
10     -0.6825165761441927    2.0
11     -0.5614332768185368    2.0
12     -0.543584499476099    2.0
13     -0.5220409195273059    2.0
14     -0.4981678674728576    2.0
15     -0.4554874247599523    2.0
16     -0.3278896331331475    2.0
17     -0.27060678100177227    2.0
18     0.2812054503635356    0.0
19     0.37888397200977053    0.0
20     0.5650539389976905    0.0
21     0.6170793022731182    0.0
22     0.6228594039763693    0.0
23     0.6851577701549856    0.0
24     0.7478470946515259    0.0
25     0.7712376131702948    0.0
26     0.9232195001096747    0.0
27     0.93786815651592    0.0
28     1.12725

In [8]:
n_elec= 6
n_orb = 5 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([12,17,18,19,20])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-228.0286641976386),
 np.float64(-7.481410269203963),
 FCIvector([[ 9.49181314e-01,  0.00000000e+00,  1.52570460e-02,
              7.81245668e-03,  7.24936010e-03,  0.00000000e+00,
              0.00000000e+00, -4.84698678e-02, -8.61860444e-04,
              0.00000000e+00],
            [ 0.00000000e+00, -1.68666192e-01,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00,  4.91496027e-02,
              5.35339399e-02,  0.00000000e+00,  0.00000000e+00,
              9.94348764e-04],
            [ 1.52570460e-02,  0.00000000e+00, -7.55503286e-02,
              9.51372714e-03,  9.76191640e-02,  0.00000000e+00,
              0.00000000e+00,  7.45568637e-04,  7.56518366e-03,
              0.00000000e+00],
            [ 7.81245668e-03,  0.00000000e+00,  9.51372714e-03,
             -4.67970924e-02,  5.43955985e-02,  0.00000000e+00,
              0.00000000e+00,  6.57103708e-03,  3.12596083e-04,
              0.00000000e+00],
            [ 7.24936010e-03,  0.00000000e

In [9]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0 1 2]      [0 1 2]      0.949181314076081
[0 1 2]      [0 2 3]      0.015257046011689
[0 1 2]      [1 2 3]      0.007812456677950
[0 1 2]      [0 1 4]      0.007249360097919
[0 1 2]      [0 3 4]      -0.048469867791946
[0 1 2]      [1 3 4]      -0.000861860444038
[0 1 3]      [0 1 3]      -0.168666191582743
[0 1 3]      [0 2 4]      0.049149602731709
[0 1 3]      [1 2 4]      0.053533939924902
[0 1 3]      [2 3 4]      0.000994348763605
[0 2 3]      [0 1 2]      0.015257046011689
[0 2 3]      [0 2 3]      -0.075550328643244
[0 2 3]      [1 2 3]      0.009513727135017
[0 2 3]      [0 1 4]      0.097619164014609
[0 2 3]      [0 3 4]      0.000745568636894
[0 2 3]      [1 3 4]      0.007565183659941
[1 2 3]      [0 1 2]      0.007812456677950
[1 2 3]      [0 2 3]      0.009513727135017
[1 2 3]      [1 2 3]      -0.046797092436189
[1 2 3]      [0 1 4]      0.054395598483280
[1 2 3]      [0 3 4]      0.006571037077406
[1 2 3]      [1 3 4]      0.000312596083105
[0 1 4]      [0 1 2]      0